# Modeling Pipelines - Predicting Student Health Risk

This notebook prepares a clean object-oriented modeling scaffold for the Playground Series S6E7 student health risk task.

The goal is to run compact field trials for CatBoost, XGBoost, LightGBM, and HistGradientBoostingClassifier with shared data preparation, validation, logging, and output saving.


# Plan

1. Resolve Kaggle and local input files.
2. Load competition data and optional original data.
3. Define reusable schema and preprocessing classes.
4. Build train, original, and test feature matrices with source-aware metadata.
5. Prepare a stratified validation plan.
6. Run model field trials in separate model blocks.
7. Save every probation submission and keep the best one as `submission.csv`.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import html
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(value):
        print(str(value)[:1200])

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## Configuration


In [ ]:
@dataclass(frozen=True)
class ModelingConfig:
    competition_slug: str = "playground-series-s6e7"
    target: str = "health_condition"
    id_column: str = "id"
    local_dataset_dir: Path = Path("../dataset")
    random_state: int = 42
    n_splits: int = 5
    output_dir: Path = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


class NotebookDisplay:
    @staticmethod
    def card(title: str, body: str):
        if HTML is None:
            print(title)
            print(body)
            return
        display(HTML(
            "<div style='border:1px solid #dbe3ef;border-radius:8px;padding:16px 18px;margin:12px 0;background:#ffffff'>"
            f"<div style='font-size:17px;font-weight:700;color:#0f172a;margin-bottom:8px'>{html.escape(title)}</div>"
            f"<div style='font-size:14px;line-height:1.55;color:#334155'>{body}</div>"
            "</div>"
        ))


config = ModelingConfig()

# Data Preparation

## Data Loading


In [ ]:
class DataLoader:
    '''Resolve competition files and optional original data without editing paths between Kaggle and local runs.'''

    def __init__(self, config: ModelingConfig):
        self.config = config

    def resolve_input_dir(self) -> Path:
        candidates = [
            Path("/kaggle/input/predicting-student-health-risk"),
            Path("/kaggle/input") / self.config.competition_slug,
            Path("/kaggle/input/playground-series-s6e7"),
            self.config.local_dataset_dir,
            Path("dataset"),
        ]
        for path in candidates:
            if (path / "train.csv").exists() and (path / "test.csv").exists():
                return path
        kaggle_root = Path("/kaggle/input")
        if kaggle_root.exists():
            for train_path in sorted(kaggle_root.rglob("train.csv")):
                path = train_path.parent
                if (path / "test.csv").exists():
                    return path
        raise FileNotFoundError("Could not find train.csv and test.csv")

    def resolve_original_path(self) -> Path | None:
        candidates = [
            Path("/kaggle/input/datasets/ziya07/college-student-health-behavior-dataset/student_health_dataset_50k.csv"),
            Path("/kaggle/input/college-student-health-behavior-dataset/student_health_dataset_50k.csv"),
            Path("/kaggle/input/student-health-dataset/student_health_dataset_50k.csv"),
            self.config.local_dataset_dir / "student_health_dataset_50k.csv",
            Path("dataset/student_health_dataset_50k.csv"),
        ]
        for path in candidates:
            if path.exists():
                return path
        kaggle_root = Path("/kaggle/input")
        if kaggle_root.exists():
            matches = sorted(kaggle_root.rglob("student_health_dataset_50k.csv"))
            if matches:
                return matches[0]
        return None

    def load(self) -> dict[str, Any]:
        input_dir = self.resolve_input_dir()
        original_path = self.resolve_original_path()
        data = {
            "input_dir": input_dir,
            "original_path": original_path,
            "train": pd.read_csv(input_dir / "train.csv"),
            "test": pd.read_csv(input_dir / "test.csv"),
            "sample_submission": pd.read_csv(input_dir / "sample_submission.csv"),
            "original": pd.read_csv(original_path) if original_path is not None else None,
        }
        return data


data = DataLoader(config).load()
train = data["train"]
test = data["test"]
sample_submission = data["sample_submission"]
original = data["original"]

NotebookDisplay.card(
    "Input resolved",
    f"Competition data: <code>{html.escape(str(data['input_dir']))}</code><br>"
    f"Train rows: <b>{len(train):,}</b>, test rows: <b>{len(test):,}</b>, sample rows: <b>{len(sample_submission):,}</b><br>"
    f"Original data: <code>{html.escape(str(data['original_path']))}</code>" if original is not None else "Original data was not found."
)

display(train.head())
display(test.head())

## Schema


In [ ]:
class SchemaInspector:
    '''Keep feature roles in one place so each future model pipeline consumes the same columns.'''

    def __init__(self, config: ModelingConfig, train: pd.DataFrame):
        self.config = config
        self.train = train

    @property
    def feature_columns(self) -> list[str]:
        return [column for column in self.train.columns if column not in {self.config.id_column, self.config.target}]

    @property
    def numeric_columns(self) -> list[str]:
        return [column for column in self.feature_columns if pd.api.types.is_numeric_dtype(self.train[column])]

    @property
    def categorical_columns(self) -> list[str]:
        return [column for column in self.feature_columns if column not in self.numeric_columns]

    def summary(self, test: pd.DataFrame) -> pd.DataFrame:
        rows = []
        for column in self.train.columns:
            rows.append({
                "column": column,
                "role": self.role(column),
                "train_dtype": str(self.train[column].dtype),
                "test_dtype": str(test[column].dtype) if column in test.columns else "-",
                "train_missing_%": self.train[column].isna().mean() * 100,
                "test_missing_%": test[column].isna().mean() * 100 if column in test.columns else np.nan,
                "train_unique": self.train[column].nunique(dropna=True),
                "test_unique": test[column].nunique(dropna=True) if column in test.columns else np.nan,
            })
        return pd.DataFrame(rows)

    def role(self, column: str) -> str:
        if column == self.config.id_column:
            return "id"
        if column == self.config.target:
            return "target"
        if column in self.numeric_columns:
            return "numeric feature"
        return "categorical feature"


schema = SchemaInspector(config, train)
display(schema.summary(test))
NotebookDisplay.card(
    "Feature roles",
    f"Numeric features: <b>{len(schema.numeric_columns)}</b><br>Categorical features: <b>{len(schema.categorical_columns)}</b>"
)

## Original Data Alignment


In [ ]:
class OriginalDataAdapter:
    '''Align the optional original dataset to the competition train schema and keep provenance explicit.'''

    def __init__(self, config: ModelingConfig, train_columns: list[str]):
        self.config = config
        self.train_columns = train_columns

    def align(self, original: pd.DataFrame | None) -> pd.DataFrame | None:
        if original is None:
            return None
        shared_columns = [column for column in self.train_columns if column in original.columns]
        aligned = original[shared_columns].copy()
        for column in self.train_columns:
            if column not in aligned.columns and column != self.config.id_column:
                aligned[column] = np.nan
        aligned = aligned[[column for column in self.train_columns if column != self.config.id_column]]
        return aligned

    def compare_target(self, train: pd.DataFrame, aligned_original: pd.DataFrame | None) -> pd.DataFrame:
        train_share = train[self.config.target].value_counts(normalize=True).mul(100).rename("train_%")
        if aligned_original is None or self.config.target not in aligned_original.columns:
            return train_share.to_frame()
        original_share = aligned_original[self.config.target].value_counts(normalize=True).mul(100).rename("original_%")
        return pd.concat([train_share, original_share], axis=1).fillna(0)


original_adapter = OriginalDataAdapter(config, list(train.columns))
aligned_original = original_adapter.align(original)
display(original_adapter.compare_target(train, aligned_original).round(3))
NotebookDisplay.card(
    "Original data policy",
    "Use competition train only for validation. Use aligned original rows only inside final training or experiments that report this choice explicitly."
)

## Preprocessing


In [ ]:
class HealthRiskFeatureEngineer:
    '''Create compact features that match the EDA findings and stay model-agnostic.'''

    numeric_base = [
        "sleep_duration",
        "heart_rate",
        "bmi",
        "calorie_expenditure",
        "step_count",
        "exercise_duration",
        "water_intake",
    ]

    categorical_base = [
        "diet_type",
        "stress_level",
        "sleep_quality",
        "physical_activity_level",
        "smoking_alcohol",
        "gender",
    ]

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        df = frame.copy()
        self.add_missing_indicators(df)
        self.add_simple_numeric_features(df)
        self.add_interaction_features(df)
        self.normalize_categoricals(df)
        return df

    def add_missing_indicators(self, df: pd.DataFrame) -> None:
        for column in self.numeric_base + self.categorical_base:
            if column in df.columns:
                df[f"{column}_is_missing"] = df[column].isna().astype("int8")

    def add_simple_numeric_features(self, df: pd.DataFrame) -> None:
        df["sleep_deprived"] = (df["sleep_duration"] < 6).astype("float")
        df["long_sleep"] = (df["sleep_duration"] >= 8).astype("float")
        df["active_minutes_per_1k_steps"] = df["exercise_duration"] / (df["step_count"] / 1000).replace(0, np.nan)
        df["calories_per_1k_steps"] = df["calorie_expenditure"] / (df["step_count"] / 1000).replace(0, np.nan)
        df["sleep_water_product"] = df["sleep_duration"] * df["water_intake"]

    def add_interaction_features(self, df: pd.DataFrame) -> None:
        df["stress_activity"] = df["stress_level"].fillna("Missing") + " | " + df["physical_activity_level"].fillna("Missing")
        df["stress_sleep_quality"] = df["stress_level"].fillna("Missing") + " | " + df["sleep_quality"].fillna("Missing")
        df["activity_smoking"] = df["physical_activity_level"].fillna("Missing") + " | " + df["smoking_alcohol"].fillna("Missing")

    def normalize_categoricals(self, df: pd.DataFrame) -> None:
        for column in self.categorical_columns(df):
            df[column] = df[column].astype("object").where(df[column].notna(), "Missing")

    def categorical_columns(self, df: pd.DataFrame) -> list[str]:
        return [column for column in df.columns if df[column].dtype == "object" and column != config.target]

    def feature_columns(self, df: pd.DataFrame) -> list[str]:
        excluded = {config.id_column, config.target}
        return [column for column in df.columns if column not in excluded]


feature_engineer = HealthRiskFeatureEngineer()
train_fe = feature_engineer.transform(train)
test_fe = feature_engineer.transform(test)
original_fe = feature_engineer.transform(aligned_original) if aligned_original is not None else None

feature_columns = feature_engineer.feature_columns(train_fe)
categorical_columns = feature_engineer.categorical_columns(train_fe[feature_columns])
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

display(pd.DataFrame({
    "feature_type": ["numeric", "categorical", "all"],
    "count": [len(numeric_columns), len(categorical_columns), len(feature_columns)],
}))
display(train_fe[feature_columns].head())

## Modeling Dataset


In [ ]:
@dataclass(frozen=True)
class ModelingDataset:
    X: pd.DataFrame
    y: pd.Series
    X_test: pd.DataFrame
    feature_columns: list[str]
    categorical_columns: list[str]
    numeric_columns: list[str]
    original_X: pd.DataFrame | None
    original_y: pd.Series | None


class ModelingDatasetBuilder:
    '''Build the shared data contract that every future model pipeline should accept.'''

    def __init__(self, config: ModelingConfig):
        self.config = config

    def build(
        self,
        train_fe: pd.DataFrame,
        test_fe: pd.DataFrame,
        original_fe: pd.DataFrame | None,
        feature_columns: list[str],
        categorical_columns: list[str],
        numeric_columns: list[str],
    ) -> ModelingDataset:
        original_X = None
        original_y = None
        if original_fe is not None and self.config.target in original_fe.columns:
            original_X = original_fe[feature_columns].copy()
            original_y = original_fe[self.config.target].copy()
        return ModelingDataset(
            X=train_fe[feature_columns].copy(),
            y=train_fe[self.config.target].copy(),
            X_test=test_fe[feature_columns].copy(),
            feature_columns=feature_columns,
            categorical_columns=categorical_columns,
            numeric_columns=numeric_columns,
            original_X=original_X,
            original_y=original_y,
        )


modeling_data = ModelingDatasetBuilder(config).build(
    train_fe,
    test_fe,
    original_fe,
    feature_columns,
    categorical_columns,
    numeric_columns,
)

display(pd.DataFrame([
    {"name": "X", "rows": len(modeling_data.X), "columns": modeling_data.X.shape[1]},
    {"name": "X_test", "rows": len(modeling_data.X_test), "columns": modeling_data.X_test.shape[1]},
    {"name": "original_X", "rows": 0 if modeling_data.original_X is None else len(modeling_data.original_X), "columns": 0 if modeling_data.original_X is None else modeling_data.original_X.shape[1]},
]))

# Validation

## Validation Plan


In [ ]:
class ValidationPlan:
    '''Create stratified folds on competition train only. Original rows stay out of validation.'''

    def __init__(self, config: ModelingConfig):
        self.config = config

    def make_folds(self, y: pd.Series) -> pd.Series:
        try:
            from sklearn.model_selection import StratifiedKFold
            splitter = StratifiedKFold(n_splits=self.config.n_splits, shuffle=True, random_state=self.config.random_state)
            folds = pd.Series(index=y.index, dtype="int64", name="fold")
            for fold, (_, valid_idx) in enumerate(splitter.split(np.zeros(len(y)), y)):
                folds.iloc[valid_idx] = fold
            return folds.astype("int64")
        except Exception:
            shuffled = y.sample(frac=1, random_state=self.config.random_state)
            folds = pd.Series(index=y.index, dtype="int64", name="fold")
            for target, index in shuffled.groupby(shuffled).groups.items():
                target_index = pd.Index(index)
                folds.loc[target_index] = np.arange(len(target_index)) % self.config.n_splits
            return folds.astype("int64")

    def fold_summary(self, y: pd.Series, folds: pd.Series) -> pd.DataFrame:
        return pd.crosstab(folds, y, normalize="index").mul(100).round(3)


validation = ValidationPlan(config)
folds = validation.make_folds(modeling_data.y)
display(validation.fold_summary(modeling_data.y, folds))
NotebookDisplay.card(
    "Validation rule",
    "Each pipeline should report fold scores on competition train only. Original data can be added to fitting folds later, but never to validation rows."
)

# Pipeline Contract


In [ ]:
@dataclass
class PipelineResult:
    name: str
    estimator: str
    config_name: str
    fold_scores: list[float]
    train_scores: list[float]
    elapsed_seconds: float
    submission_path: Path | None = None
    oof_predictions: pd.Series | None = None
    test_predictions: pd.Series | None = None
    notes: str = ""

    @property
    def mean_score(self) -> float:
        return float(np.mean(self.fold_scores)) if self.fold_scores else np.nan

    @property
    def std_score(self) -> float:
        return float(np.std(self.fold_scores)) if self.fold_scores else np.nan

    @property
    def mean_train_score(self) -> float:
        return float(np.mean(self.train_scores)) if self.train_scores else np.nan


class ComparisonBoard:
    '''Store model results in one small table so experiments stay comparable.'''

    def __init__(self):
        self.results: list[PipelineResult] = []

    def add(self, result: PipelineResult) -> None:
        self.results = [existing for existing in self.results if existing.name != result.name]
        self.results.append(result)

    def table(self) -> pd.DataFrame:
        rows = []
        for result in self.results:
            rows.append({
                "pipeline": result.name,
                "estimator": result.estimator,
                "config": result.config_name,
                "mean_score": result.mean_score,
                "std_score": result.std_score,
                "mean_train_score": result.mean_train_score,
                "elapsed_seconds": result.elapsed_seconds,
                "submission_path": None if result.submission_path is None else str(result.submission_path),
                "notes": result.notes,
            })
        columns = ["pipeline", "estimator", "config", "mean_score", "std_score", "mean_train_score", "elapsed_seconds", "submission_path", "notes"]
        return pd.DataFrame(rows, columns=columns).sort_values("mean_score", ascending=False, na_position="last") if rows else pd.DataFrame(columns=columns)


comparison = ComparisonBoard()
display(comparison.table())
NotebookDisplay.card(
    "Pipeline contract",
    "Each field trial will return fold scores, train scores, elapsed time, OOF predictions, test predictions, and a saved submission."
)



# Field Trials

This section runs small probation pipelines. Each estimator is tested on three compact configurations so the notebook can compare directionally useful models before spending time on full tuning.

The best submission is written to `submission.csv`. Every trial submission is also kept under `probations/`.


## Trial Configuration


In [ ]:
@dataclass(frozen=True)
class TrialConfig:
    name: str
    use_original: bool
    class_weight: str | None
    model_size: str
    max_train_seconds: int = 120


trial_configs = [
    TrialConfig(name="small", use_original=False, class_weight=None, model_size="small"),
    TrialConfig(name="balanced", use_original=False, class_weight="balanced", model_size="small"),
    TrialConfig(name="balanced_original", use_original=True, class_weight="balanced", model_size="small"),
]


class TrialLogger:
    '''Print compact fold logs that are readable in Kaggle output.'''

    def start_trial(self, estimator: str, config_name: str) -> None:
        print()
        print(f"Estimator: {estimator} | config: {config_name}")
        print("=" * 72)

    def skip_trial(self, estimator: str, reason: str) -> None:
        print(f"Estimator: {estimator} is skipped")
        print(f"Reason: {reason}")

    def fold_done(self, estimator: str, fold: int, train_score: float, val_score: float, elapsed: float) -> None:
        print(f"Estimator: {estimator} of fold {fold} is fitted")
        print(f"Train balanced accuracy score: {train_score}")
        print(f"Val balanced accuracy score: {val_score}")
        print(f"Elapsed time: {elapsed:.0f} [s]")
        print("-" * 72)

    def trial_done(self, name: str, mean_score: float, std_score: float, elapsed: float) -> None:
        print(f"Trial: {name} is finished")
        print(f"Mean validation balanced accuracy: {mean_score}")
        print(f"Std validation balanced accuracy: {std_score}")
        print(f"Total elapsed time: {elapsed:.0f} [s]")
        print("=" * 72)


logger = TrialLogger()



## Trial Preprocessing


In [ ]:
class TrialPreprocessor:
    '''Prepare model-specific matrices while keeping the same fold split and feature set.'''

    def __init__(self, data: ModelingDataset):
        self.data = data

    def catboost(self, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame, X_original: pd.DataFrame | None):
        return self.fill_categories(X_train), self.fill_categories(X_valid), self.fill_categories(X_test), None if X_original is None else self.fill_categories(X_original)

    def lightgbm(self, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame, X_original: pd.DataFrame | None):
        frames = [self.fill_categories(X_train), self.fill_categories(X_valid), self.fill_categories(X_test)]
        if X_original is not None:
            frames.append(self.fill_categories(X_original))
        for frame in frames:
            for column in self.data.categorical_columns:
                frame[column] = frame[column].astype("category")
        if X_original is None:
            return frames[0], frames[1], frames[2], None
        return frames[0], frames[1], frames[2], frames[3]

    def encoded(self, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame, X_original: pd.DataFrame | None):
        from sklearn.preprocessing import OrdinalEncoder
        X_train_out = X_train.copy()
        X_valid_out = X_valid.copy()
        X_test_out = X_test.copy()
        X_original_out = None if X_original is None else X_original.copy()
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        train_categories = X_train_out[self.data.categorical_columns].astype("object").fillna("Missing")
        encoder.fit(train_categories)
        for frame in [X_train_out, X_valid_out, X_test_out] + ([] if X_original_out is None else [X_original_out]):
            frame[self.data.categorical_columns] = encoder.transform(frame[self.data.categorical_columns].astype("object").fillna("Missing"))
        for column in self.data.numeric_columns:
            median = X_train_out[column].median()
            X_train_out[column] = X_train_out[column].fillna(median)
            X_valid_out[column] = X_valid_out[column].fillna(median)
            X_test_out[column] = X_test_out[column].fillna(median)
            if X_original_out is not None:
                X_original_out[column] = X_original_out[column].fillna(median)
        return X_train_out, X_valid_out, X_test_out, X_original_out

    def fill_categories(self, frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        for column in self.data.categorical_columns:
            out[column] = out[column].astype("object").where(out[column].notna(), "Missing")
        return out


## Estimator Adapters


In [ ]:
class BaseEstimatorAdapter:
    name = "base"

    def available(self) -> tuple[bool, str]:
        return True, "available"

    def prepare(self, preprocessor: TrialPreprocessor, X_train, X_valid, X_test, X_original):
        return X_train, X_valid, X_test, X_original

    def build_model(self, trial_config: TrialConfig, random_state: int):
        raise NotImplementedError

    def fit(self, model, X_train, y_train, X_valid, y_valid, sample_weight):
        model.fit(X_train, y_train, sample_weight=sample_weight)
        return model

    def predict(self, model, X):
        return model.predict(X)

    def predict_proba(self, model, X):
        if hasattr(model, "predict_proba"):
            return model.predict_proba(X)
        predictions = np.asarray(model.predict(X)).astype(int)
        proba = np.zeros((len(predictions), len(np.unique(predictions))))
        proba[np.arange(len(predictions)), predictions] = 1
        return proba


class CatBoostAdapter(BaseEstimatorAdapter):
    name = "catboost"

    def available(self) -> tuple[bool, str]:
        try:
            import catboost
            return True, catboost.__version__
        except Exception as exc:
            return False, f"catboost import failed: {type(exc).__name__}"

    def prepare(self, preprocessor: TrialPreprocessor, X_train, X_valid, X_test, X_original):
        return preprocessor.catboost(X_train, X_valid, X_test, X_original)

    def build_model(self, trial_config: TrialConfig, random_state: int):
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric="MultiClass",
            iterations=900,
            learning_rate=0.06,
            depth=8,
            l2_leaf_reg=5,
            random_seed=random_state,
            task_type="GPU",
            devices="0:1",
            verbose=False,
            allow_writing_files=False,
            early_stopping_rounds=120,
        )

    def fit(self, model, X_train, y_train, X_valid, y_valid, sample_weight):
        from catboost import CatBoostClassifier, Pool
        cat_features = [X_train.columns.get_loc(column) for column in modeling_data.categorical_columns]
        train_pool = Pool(X_train, y_train, cat_features=cat_features, weight=sample_weight)
        valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)
        try:
            model.fit(train_pool, eval_set=valid_pool)
        except Exception as exc:
            message = str(exc).lower()
            if "cuda" not in message and "gpu" not in message:
                raise
            cpu_params = model.get_params()
            cpu_params["task_type"] = "CPU"
            cpu_params.pop("devices", None)
            model = CatBoostClassifier(**cpu_params)
            model.fit(train_pool, eval_set=valid_pool)
        return model


class LightGBMAdapter(BaseEstimatorAdapter):
    name = "lightgbm"

    def available(self) -> tuple[bool, str]:
        try:
            import lightgbm
            return True, lightgbm.__version__
        except Exception as exc:
            return False, f"lightgbm import failed: {type(exc).__name__}"

    def prepare(self, preprocessor: TrialPreprocessor, X_train, X_valid, X_test, X_original):
        return preprocessor.lightgbm(X_train, X_valid, X_test, X_original)

    def build_model(self, trial_config: TrialConfig, random_state: int):
        from lightgbm import LGBMClassifier
        return LGBMClassifier(
            objective="multiclass",
            n_estimators=900,
            learning_rate=0.05,
            num_leaves=63,
            max_depth=-1,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=random_state,
            device_type="gpu",
            verbosity=-1,
        )

    def fit(self, model, X_train, y_train, X_valid, y_valid, sample_weight):
        try:
            from lightgbm import early_stopping, log_evaluation
            model.fit(
                X_train,
                y_train,
                sample_weight=sample_weight,
                eval_set=[(X_valid, y_valid)],
                categorical_feature=modeling_data.categorical_columns,
                callbacks=[early_stopping(120, verbose=False), log_evaluation(0)],
            )
        except Exception:
            model.set_params(device_type="cpu")
            model.fit(X_train, y_train, sample_weight=sample_weight, categorical_feature=modeling_data.categorical_columns)
        return model


class XGBoostAdapter(BaseEstimatorAdapter):
    name = "xgboost"

    def available(self) -> tuple[bool, str]:
        try:
            import xgboost
            return True, xgboost.__version__
        except Exception as exc:
            return False, f"xgboost import failed: {type(exc).__name__}"

    def prepare(self, preprocessor: TrialPreprocessor, X_train, X_valid, X_test, X_original):
        return preprocessor.encoded(X_train, X_valid, X_test, X_original)

    def build_model(self, trial_config: TrialConfig, random_state: int):
        from xgboost import XGBClassifier
        return XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_estimators=700,
            learning_rate=0.06,
            max_depth=7,
            min_child_weight=4,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=random_state,
            tree_method="hist",
            device="cuda",
        )

    def fit(self, model, X_train, y_train, X_valid, y_valid, sample_weight):
        try:
            model.fit(X_train, y_train, sample_weight=sample_weight, eval_set=[(X_valid, y_valid)], verbose=False)
        except Exception:
            model.set_params(device="cpu")
            model.fit(X_train, y_train, sample_weight=sample_weight, eval_set=[(X_valid, y_valid)], verbose=False)
        return model


class HGBCAdapter(BaseEstimatorAdapter):
    name = "hgbc"

    def available(self) -> tuple[bool, str]:
        try:
            import sklearn
            return True, sklearn.__version__
        except Exception as exc:
            return False, f"sklearn import failed: {type(exc).__name__}"

    def prepare(self, preprocessor: TrialPreprocessor, X_train, X_valid, X_test, X_original):
        return preprocessor.encoded(X_train, X_valid, X_test, X_original)

    def build_model(self, trial_config: TrialConfig, random_state: int):
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(
            learning_rate=0.08,
            max_iter=280,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )



## Field Trial Runner


In [ ]:
class FieldTrialRunner:
    '''Run every estimator on every probation config and save all submissions.'''

    def __init__(self, config: ModelingConfig, data: ModelingDataset, folds: pd.Series, sample_submission: pd.DataFrame, logger: TrialLogger):
        self.config = config
        self.data = data
        self.folds = folds
        self.sample_submission = sample_submission
        self.logger = logger
        self.preprocessor = TrialPreprocessor(data)
        self.probation_dir = config.output_dir / "probations"
        self.probation_dir.mkdir(parents=True, exist_ok=True)

    def run(self, adapters: list[BaseEstimatorAdapter], trial_configs: list[TrialConfig], comparison: ComparisonBoard) -> pd.DataFrame:
        for adapter in adapters:
            available, reason = adapter.available()
            if not available:
                self.logger.skip_trial(adapter.name, reason)
                continue
            for trial_config in trial_configs:
                result = self.run_one(adapter, trial_config)
                comparison.add(result)
                display(comparison.table())
        self.save_best(comparison)
        return comparison.table()

    def run_one(self, adapter: BaseEstimatorAdapter, trial_config: TrialConfig) -> PipelineResult:
        from sklearn.metrics import balanced_accuracy_score
        from sklearn.preprocessing import LabelEncoder
        from sklearn.utils.class_weight import compute_sample_weight
        label_encoder = LabelEncoder()
        y_encoded = pd.Series(label_encoder.fit_transform(self.data.y), index=self.data.y.index, name=self.data.y.name)
        original_y_encoded = None if self.data.original_y is None else pd.Series(label_encoder.transform(self.data.original_y), index=self.data.original_y.index, name=self.data.original_y.name)
        fold_scores = []
        train_scores = []
        oof_proba = np.zeros((len(self.data.X), len(label_encoder.classes_)))
        test_proba = np.zeros((len(self.data.X_test), len(label_encoder.classes_)))
        trial_start = time.time()
        self.logger.start_trial(adapter.name, trial_config.name)
        for fold in sorted(self.folds.unique()):
            fold_start = time.time()
            train_idx = self.folds[self.folds != fold].index
            valid_idx = self.folds[self.folds == fold].index
            X_train = self.data.X.loc[train_idx]
            y_train = y_encoded.loc[train_idx]
            X_valid = self.data.X.loc[valid_idx]
            y_valid = y_encoded.loc[valid_idx]
            X_original = self.data.original_X if trial_config.use_original else None
            y_original = original_y_encoded if trial_config.use_original else None
            X_train, X_valid, X_test, X_original = adapter.prepare(self.preprocessor, X_train, X_valid, self.data.X_test, X_original)
            if X_original is not None and y_original is not None:
                X_train = pd.concat([X_train, X_original], axis=0, ignore_index=True)
                y_train = pd.concat([y_train.reset_index(drop=True), y_original.reset_index(drop=True)], axis=0, ignore_index=True)
            sample_weight = compute_sample_weight("balanced", y_train) if trial_config.class_weight == "balanced" else None
            model = adapter.build_model(trial_config, self.config.random_state + int(fold))
            model = adapter.fit(model, X_train, y_train, X_valid, y_valid, sample_weight)
            train_pred = np.asarray(adapter.predict(model, X_train)).reshape(-1).astype(int)
            valid_pred = np.asarray(adapter.predict(model, X_valid)).reshape(-1).astype(int)
            train_score = balanced_accuracy_score(y_train, train_pred)
            valid_score = balanced_accuracy_score(y_valid, valid_pred)
            valid_proba = self.aligned_proba(adapter.predict_proba(model, X_valid), model, label_encoder)
            fold_test_proba = self.aligned_proba(adapter.predict_proba(model, X_test), model, label_encoder)
            oof_proba[valid_idx] = valid_proba
            test_proba += fold_test_proba / self.config.n_splits
            fold_scores.append(float(valid_score))
            train_scores.append(float(train_score))
            self.logger.fold_done(adapter.name, int(fold), float(train_score), float(valid_score), time.time() - fold_start)
        prediction_codes = test_proba.argmax(axis=1)
        test_predictions = pd.Series(label_encoder.inverse_transform(prediction_codes), index=self.data.X_test.index, name=self.config.target)
        oof_predictions = pd.Series(label_encoder.inverse_transform(oof_proba.argmax(axis=1)), index=self.data.X.index, name=self.config.target)
        elapsed = time.time() - trial_start
        result = PipelineResult(
            name=f"{adapter.name}_{trial_config.name}",
            estimator=adapter.name,
            config_name=trial_config.name,
            fold_scores=fold_scores,
            train_scores=train_scores,
            elapsed_seconds=elapsed,
            oof_predictions=oof_predictions,
            test_predictions=test_predictions,
            notes=f"class_weight={trial_config.class_weight}; use_original={trial_config.use_original}",
        )
        result.submission_path = self.save_submission(result)
        self.logger.trial_done(result.name, result.mean_score, result.std_score, elapsed)
        return result

    def aligned_proba(self, proba: np.ndarray, model, label_encoder) -> np.ndarray:
        proba = np.asarray(proba)
        if proba.shape[1] == len(label_encoder.classes_):
            return proba
        aligned = np.zeros((proba.shape[0], len(label_encoder.classes_)))
        classes = getattr(model, "classes_", np.arange(proba.shape[1]))
        for source_idx, class_value in enumerate(classes):
            aligned[:, int(class_value)] = proba[:, source_idx]
        return aligned

    def save_submission(self, result: PipelineResult) -> Path:
        submission = self.sample_submission.copy()
        submission[self.config.target] = result.test_predictions.to_numpy()
        path = self.probation_dir / f"{result.name}.csv"
        submission.to_csv(path, index=False)
        return path

    def save_best(self, comparison: ComparisonBoard) -> None:
        table = comparison.table()
        if table.empty:
            return
        best_name = table.iloc[0]["pipeline"]
        best = next(result for result in comparison.results if result.name == best_name)
        submission = self.sample_submission.copy()
        submission[self.config.target] = best.test_predictions.to_numpy()
        best_path = self.config.output_dir / "submission.csv"
        submission.to_csv(best_path, index=False)
        NotebookDisplay.card(
            "Best submission saved",
            f"Best pipeline: <b>{html.escape(best.name)}</b><br>CV balanced accuracy: <b>{best.mean_score:.6f}</b><br>Root submission: <code>{html.escape(str(best_path))}</code><br>All submissions: <code>{html.escape(str(self.probation_dir))}</code>"
        )


In [ ]:
runner = FieldTrialRunner(config, modeling_data, folds, sample_submission, logger)


## CatBoost Field Trials


In [ ]:
catboost_summary = runner.run([CatBoostAdapter()], trial_configs, comparison)
display(catboost_summary)


## XGBoost Field Trials


In [ ]:
xgboost_summary = runner.run([XGBoostAdapter()], trial_configs, comparison)
display(xgboost_summary)


## LightGBM Field Trials


In [ ]:
lightgbm_summary = runner.run([LightGBMAdapter()], trial_configs, comparison)
display(lightgbm_summary)


## HGBC Field Trials


In [ ]:
hgbc_summary = runner.run([HGBCAdapter()], trial_configs, comparison)
display(hgbc_summary)


## Final Trial Summary


In [ ]:
trial_summary = comparison.table()
display(trial_summary)
runner.save_best(comparison)


# Competitive Pipeline

This block is the stronger experiment path. It keeps validation fold-safe, adds target-statistical encodings fitted inside each fold, trains tuned HGBC and CatBoost models, and blends their probabilities by OOF balanced accuracy.


## Competitive Target Encoding


In [ ]:
class FoldSafeTargetEncoder:
    '''Create class-probability encodings from training-fold rows only.'''

    def __init__(self, categorical_columns: list[str], n_classes: int, smoothing: float = 30.0):
        self.categorical_columns = categorical_columns
        self.n_classes = n_classes
        self.smoothing = smoothing
        self.global_probs: dict[int, float] = {}
        self.category_maps: dict[str, dict[int, dict[Any, float]]] = {}
        self.frequency_maps: dict[str, dict[Any, float]] = {}
        self.output_columns: list[str] = []

    def fit(self, X: pd.DataFrame, y: pd.Series):
        y = pd.Series(y).reset_index(drop=True)
        X_work = X.reset_index(drop=True)
        self.global_probs = {class_id: float((y == class_id).mean()) for class_id in range(self.n_classes)}
        self.category_maps = {}
        self.frequency_maps = {}
        self.output_columns = []
        for column in self.categorical_columns:
            values = X_work[column].astype("object").fillna("Missing")
            counts = values.value_counts(dropna=False)
            self.frequency_maps[column] = (counts / len(values)).to_dict()
            self.category_maps[column] = {}
            for class_id in range(self.n_classes):
                class_counts = values[y == class_id].value_counts(dropna=False)
                smoothed = (class_counts.reindex(counts.index, fill_value=0) + self.global_probs[class_id] * self.smoothing) / (counts + self.smoothing)
                self.category_maps[column][class_id] = smoothed.to_dict()
                self.output_columns.append(f"{column}_te_class_{class_id}")
            self.output_columns.append(f"{column}_frequency")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        out = X.copy()
        for column in self.categorical_columns:
            values = out[column].astype("object").fillna("Missing")
            for class_id in range(self.n_classes):
                out[f"{column}_te_class_{class_id}"] = values.map(self.category_maps[column][class_id]).fillna(self.global_probs[class_id]).astype("float32")
            out[f"{column}_frequency"] = values.map(self.frequency_maps[column]).fillna(0).astype("float32")
        return out


class CompetitivePreprocessor:
    '''Build fold-safe encoded matrices for strong tree models.'''

    def __init__(self, data: ModelingDataset, n_classes: int):
        self.data = data
        self.n_classes = n_classes

    def build_fold(self, X_train: pd.DataFrame, y_train: pd.Series, X_valid: pd.DataFrame, X_test: pd.DataFrame, X_original: pd.DataFrame | None, y_original: pd.Series | None):
        fit_X = X_train
        fit_y = y_train
        if X_original is not None and y_original is not None:
            fit_X = pd.concat([X_train, X_original], axis=0, ignore_index=True)
            fit_y = pd.concat([y_train.reset_index(drop=True), y_original.reset_index(drop=True)], axis=0, ignore_index=True)
        encoder = FoldSafeTargetEncoder(self.data.categorical_columns, self.n_classes).fit(fit_X, fit_y)
        X_train_te = encoder.transform(fit_X)
        X_valid_te = encoder.transform(X_valid)
        X_test_te = encoder.transform(X_test)
        return X_train_te, fit_y, X_valid_te, X_test_te, encoder.output_columns

    def for_catboost(self, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame):
        return self.fill_categories(X_train), self.fill_categories(X_valid), self.fill_categories(X_test)

    def for_hgbc(self, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame):
        from sklearn.preprocessing import OrdinalEncoder
        X_train_out = X_train.copy()
        X_valid_out = X_valid.copy()
        X_test_out = X_test.copy()
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        encoder.fit(X_train_out[self.data.categorical_columns].astype("object").fillna("Missing"))
        for frame in [X_train_out, X_valid_out, X_test_out]:
            frame[self.data.categorical_columns] = encoder.transform(frame[self.data.categorical_columns].astype("object").fillna("Missing"))
        for column in X_train_out.columns:
            if column not in self.data.categorical_columns:
                median = X_train_out[column].median()
                X_train_out[column] = X_train_out[column].fillna(median)
                X_valid_out[column] = X_valid_out[column].fillna(median)
                X_test_out[column] = X_test_out[column].fillna(median)
        return X_train_out, X_valid_out, X_test_out

    def fill_categories(self, frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        for column in self.data.categorical_columns:
            out[column] = out[column].astype("object").where(out[column].notna(), "Missing")
        return out


## Competitive Estimators


In [ ]:
class CompetitiveHGBCAdapter:
    name = "competitive_hgbc"

    def available(self) -> tuple[bool, str]:
        try:
            import sklearn
            return True, sklearn.__version__
        except Exception as exc:
            return False, f"sklearn import failed: {type(exc).__name__}"

    def prepare(self, preprocessor: CompetitivePreprocessor, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame):
        return preprocessor.for_hgbc(X_train, X_valid, X_test)

    def build_model(self, random_state: int):
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(
            scoring="balanced_accuracy",
            class_weight="balanced",
            max_iter=10000,
            n_iter_no_change=20,
            validation_fraction=0.1,
            learning_rate=0.05,
            max_leaf_nodes=23,
            min_samples_leaf=20,
            l2_regularization=0.0,
            max_depth=None,
            random_state=random_state,
        )

    def fit(self, model, X_train: pd.DataFrame, y_train: pd.Series, X_valid: pd.DataFrame, y_valid: pd.Series):
        model.fit(X_train, y_train)
        return model


class CompetitiveCatBoostAdapter:
    name = "competitive_catboost"

    def available(self) -> tuple[bool, str]:
        try:
            import catboost
            return True, catboost.__version__
        except Exception as exc:
            return False, f"catboost import failed: {type(exc).__name__}"

    def prepare(self, preprocessor: CompetitivePreprocessor, X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame):
        return preprocessor.for_catboost(X_train, X_valid, X_test)

    def build_model(self, random_state: int):
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric="MultiClass",
            auto_class_weights="Balanced",
            n_estimators=10000,
            learning_rate=0.015,
            max_depth=6,
            early_stopping_rounds=150,
            l2_leaf_reg=9,
            min_data_in_leaf=1,
            bagging_temperature=1.0,
            random_strength=1.0,
            max_bin=1024,
            task_type="GPU",
            devices="0:1",
            verbose=False,
            allow_writing_files=False,
            random_state=random_state,
        )

    def fit(self, model, X_train: pd.DataFrame, y_train: pd.Series, X_valid: pd.DataFrame, y_valid: pd.Series):
        from catboost import CatBoostClassifier, Pool
        cat_features = [X_train.columns.get_loc(column) for column in modeling_data.categorical_columns]
        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)
        try:
            model.fit(train_pool, eval_set=valid_pool)
        except Exception as exc:
            message = str(exc).lower()
            if "cuda" not in message and "gpu" not in message:
                raise
            params = model.get_params()
            params["task_type"] = "CPU"
            params.pop("devices", None)
            model = CatBoostClassifier(**params)
            model.fit(train_pool, eval_set=valid_pool)
        return model


## Competitive Runner


In [ ]:
class CompetitiveRunner:
    '''Train strong fold-safe models, save individual submissions, and save an OOF-weighted blend.'''

    def __init__(self, config: ModelingConfig, data: ModelingDataset, folds: pd.Series, sample_submission: pd.DataFrame, logger: TrialLogger):
        self.config = config
        self.data = data
        self.folds = folds
        self.sample_submission = sample_submission
        self.logger = logger
        self.probation_dir = config.output_dir / "probations"
        self.probation_dir.mkdir(parents=True, exist_ok=True)

    def run(self, adapters: list[Any], comparison: ComparisonBoard) -> pd.DataFrame:
        from sklearn.metrics import balanced_accuracy_score
        from sklearn.preprocessing import LabelEncoder
        label_encoder = LabelEncoder()
        y_encoded = pd.Series(label_encoder.fit_transform(self.data.y), index=self.data.y.index, name=self.data.y.name)
        original_y_encoded = None if self.data.original_y is None else pd.Series(label_encoder.transform(self.data.original_y), index=self.data.original_y.index, name=self.data.original_y.name)
        preprocessor = CompetitivePreprocessor(self.data, len(label_encoder.classes_))
        val_prob_list = []
        test_prob_list = []
        result_list = []
        for adapter in adapters:
            available, reason = adapter.available()
            if not available:
                self.logger.skip_trial(adapter.name, reason)
                continue
            result, val_proba, test_proba = self.run_estimator(adapter, preprocessor, y_encoded, original_y_encoded, label_encoder)
            comparison.add(result)
            result_list.append(result)
            val_prob_list.append(val_proba)
            test_prob_list.append(test_proba)
            display(comparison.table())
        if val_prob_list:
            blend_result = self.blend_results(result_list, val_prob_list, test_prob_list, y_encoded, label_encoder, balanced_accuracy_score)
            comparison.add(blend_result)
            self.save_best(comparison)
        return comparison.table()

    def run_estimator(self, adapter: Any, preprocessor: CompetitivePreprocessor, y_encoded: pd.Series, original_y_encoded: pd.Series | None, label_encoder: Any):
        from sklearn.metrics import balanced_accuracy_score
        fold_scores = []
        train_scores = []
        val_proba = np.zeros((len(self.data.X), len(label_encoder.classes_)), dtype="float32")
        test_proba = np.zeros((len(self.data.X_test), len(label_encoder.classes_)), dtype="float32")
        trial_start = time.time()
        self.logger.start_trial(adapter.name, "competitive")
        for fold in sorted(self.folds.unique()):
            fold_start = time.time()
            train_idx = self.folds[self.folds != fold].index
            valid_idx = self.folds[self.folds == fold].index
            X_train, y_train, X_valid, X_test, _ = preprocessor.build_fold(
                self.data.X.loc[train_idx],
                y_encoded.loc[train_idx],
                self.data.X.loc[valid_idx],
                self.data.X_test,
                self.data.original_X,
                original_y_encoded,
            )
            y_valid = y_encoded.loc[valid_idx]
            X_train, X_valid, X_test = adapter.prepare(preprocessor, X_train, X_valid, X_test)
            model = adapter.build_model(self.config.random_state + int(fold))
            model = adapter.fit(model, X_train, y_train, X_valid, y_valid)
            train_pred = np.asarray(model.predict(X_train)).reshape(-1).astype(int)
            valid_pred = np.asarray(model.predict(X_valid)).reshape(-1).astype(int)
            train_score = balanced_accuracy_score(y_train, train_pred)
            valid_score = balanced_accuracy_score(y_valid, valid_pred)
            val_proba[valid_idx] = self.aligned_proba(model.predict_proba(X_valid), model, label_encoder)
            test_proba += self.aligned_proba(model.predict_proba(X_test), model, label_encoder) / self.config.n_splits
            fold_scores.append(float(valid_score))
            train_scores.append(float(train_score))
            self.logger.fold_done(adapter.name, int(fold), float(train_score), float(valid_score), time.time() - fold_start)
        elapsed = time.time() - trial_start
        predictions = pd.Series(label_encoder.inverse_transform(test_proba.argmax(axis=1)), index=self.data.X_test.index, name=self.config.target)
        oof_predictions = pd.Series(label_encoder.inverse_transform(val_proba.argmax(axis=1)), index=self.data.X.index, name=self.config.target)
        result = PipelineResult(
            name=adapter.name,
            estimator=adapter.name,
            config_name="competitive",
            fold_scores=fold_scores,
            train_scores=train_scores,
            elapsed_seconds=elapsed,
            oof_predictions=oof_predictions,
            test_predictions=predictions,
            notes="fold-safe target encoding; original data in train folds",
        )
        result.submission_path = self.save_submission(result)
        self.logger.trial_done(result.name, result.mean_score, result.std_score, elapsed)
        return result, val_proba, test_proba

    def blend_results(self, results: list[PipelineResult], val_prob_list: list[np.ndarray], test_prob_list: list[np.ndarray], y_encoded: pd.Series, label_encoder: Any, scorer: Any) -> PipelineResult:
        val_stack = np.stack(val_prob_list)
        test_stack = np.stack(test_prob_list)
        candidates = [np.ones(len(results)) / len(results)]
        for index in range(len(results)):
            weight = np.zeros(len(results))
            weight[index] = 1.0
            candidates.append(weight)
        if len(results) == 2:
            candidates.extend(np.array([w, 1 - w]) for w in np.linspace(0, 1, 101))
        rng = np.random.default_rng(self.config.random_state)
        candidates.extend(rng.dirichlet(np.ones(len(results)), size=300))
        best_score = -1.0
        best_weights = candidates[0]
        best_val = None
        for weights in candidates:
            val_blend = np.tensordot(weights, val_stack, axes=(0, 0))
            score = scorer(y_encoded, val_blend.argmax(axis=1))
            if score > best_score:
                best_score = float(score)
                best_weights = weights
                best_val = val_blend
        test_blend = np.tensordot(best_weights, test_stack, axes=(0, 0))
        predictions = pd.Series(label_encoder.inverse_transform(test_blend.argmax(axis=1)), index=self.data.X_test.index, name=self.config.target)
        oof_predictions = pd.Series(label_encoder.inverse_transform(best_val.argmax(axis=1)), index=self.data.X.index, name=self.config.target)
        result = PipelineResult(
            name="competitive_probability_blend",
            estimator="blend",
            config_name="competitive",
            fold_scores=[best_score],
            train_scores=[],
            elapsed_seconds=sum(result.elapsed_seconds for result in results),
            oof_predictions=oof_predictions,
            test_predictions=predictions,
            notes="weights=" + ", ".join(f"{result.name}:{weight:.3f}" for result, weight in zip(results, best_weights)),
        )
        result.submission_path = self.save_submission(result)
        print("Competitive blend is fitted")
        print(f"OOF balanced accuracy score: {best_score}")
        print(result.notes)
        return result

    def aligned_proba(self, proba: np.ndarray, model: Any, label_encoder: Any) -> np.ndarray:
        proba = np.asarray(proba)
        if proba.shape[1] == len(label_encoder.classes_):
            return proba
        aligned = np.zeros((proba.shape[0], len(label_encoder.classes_)), dtype="float32")
        classes = getattr(model, "classes_", np.arange(proba.shape[1]))
        for source_idx, class_value in enumerate(classes):
            aligned[:, int(class_value)] = proba[:, source_idx]
        return aligned

    def save_submission(self, result: PipelineResult) -> Path:
        submission = self.sample_submission.copy()
        submission[self.config.target] = result.test_predictions.to_numpy()
        path = self.probation_dir / f"{result.name}.csv"
        submission.to_csv(path, index=False)
        return path

    def save_best(self, comparison: ComparisonBoard) -> None:
        table = comparison.table()
        if table.empty:
            return
        best_name = table.iloc[0]["pipeline"]
        best = next(result for result in comparison.results if result.name == best_name)
        submission = self.sample_submission.copy()
        submission[self.config.target] = best.test_predictions.to_numpy()
        best_path = self.config.output_dir / "submission.csv"
        submission.to_csv(best_path, index=False)
        NotebookDisplay.card(
            "Best submission saved",
            f"Best pipeline: <b>{html.escape(best.name)}</b><br>CV balanced accuracy: <b>{best.mean_score:.6f}</b><br>Root submission: <code>{html.escape(str(best_path))}</code><br>All submissions: <code>{html.escape(str(self.probation_dir))}</code>"
        )


## Run Competitive Pipeline


In [ ]:
competitive_runner = CompetitiveRunner(config, modeling_data, folds, sample_submission, logger)
competitive_summary = competitive_runner.run(
    [CompetitiveHGBCAdapter(), CompetitiveCatBoostAdapter()],
    comparison,
)
display(competitive_summary)


# Output Policy

The root `submission.csv` is always produced from the best validation pipeline. The `probations/` directory keeps every trial submission so public leaderboard checks can be traced back to the exact pipeline configuration.
